<div style="text-align: center;">
<pre style="display: inline-block; text-align: left;">
Fichiers SCOPUS txt
        ↓
Extraction des articles
        ↓
Pour chaque article :
    titre + abstract + keywords
        ↓
Prompt envoyé à Qwen/Ollama
        ↓
Réponse JSON : include / exclude / uncertain
        ↓
Sauvegarde checkpoint
        ↓
Export final Excel + CSV
</pre>
</div>

In [1]:
import re
import json
import time
import requests
import pandas as pd
from pathlib import Path


start = time.perf_counter()

MODEL = "qwen3:8b"

FILES = ["scopus_results.txt"]

OUTPUT_EXCEL = "screening_results_scopus.xlsx"
OUTPUT_CSV = "screening_results_scopus.csv"

CHECKPOINT_CSV = "screening_checkpoint_scopus.csv"
CHECKPOINT_EXCEL = "screening_checkpoint_scopus.xlsx"

TIMEOUT_SECONDS = 180


INCLUSION_CRITERIA = """
Include only original research papers related to visuo-haptic / visuo-tactile perception.

The paper should involve at least two modalities among:
- vision / visual / RGB / camera
- tactile / touch / haptic / force / pressure / texture sensing

The paper should focus on perception-level tasks such as:
- object recognition
- object classification
- material recognition
- texture recognition or analysis
- surface property recognition
- 3D shape recognition
- object property or attribute recognition
- multimodal or cross-modal representation learning
- visual-tactile or visual-haptic fusion
"""

EXCLUSION_CRITERIA = """
Exclude papers if:
- the main focus is grasping, grasp planning, manipulation, robot control, trajectory planning, or pose estimation
- the paper is mainly about teleoperation, VR user study, haptic rendering, or human perception without machine perception
- the paper is a review, survey, tutorial, editorial, or non-original study
- the abstract does not clearly involve both visual and tactile/haptic information
- the task is not related to perception or recognition
"""


def clean_text(x):
    if not x:
        return ""
    x = x.replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def split_scopus_txt(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\xa0", " ")

    chunks = re.split(
        r"\n\s*(?=[A-Z][^\n]+?\nAUTHOR FULL NAMES:)",
        text
    )

    chunks = [
        c.strip()
        for c in chunks
        if "AUTHOR FULL NAMES:" in c and "ABSTRACT:" in c
    ]

    print("Detected article chunks:", len(chunks))
    print("Detected ABSTRACT fields:", len(re.findall(r"\bABSTRACT:", text)))
    print("Detected DOI fields:", len(re.findall(r"\bDOI:", text)))
    print(
        "Detected Scopus links:",
        len(re.findall(r"https://www\.scopus\.com/pages/publications/", text))
    )

    papers = []

    for chunk in chunks:
        lines = [l.strip() for l in chunk.split("\n") if l.strip()]

        authors = lines[0] if lines else ""

        title = ""
        for i, line in enumerate(lines):
            if line.startswith("AUTHOR FULL NAMES:"):
                if i + 2 < len(lines):
                    title = lines[i + 2]
                break

        source_match = re.search(
            r"\n\((\d{4})\).*?(?=\nDOI:|\nhttps://|\nABSTRACT:)",
            chunk,
            flags=re.DOTALL
        )
        source_info = clean_text(source_match.group(0)) if source_match else ""

        doi_match = re.search(r"\bDOI:\s*(.+)", chunk)
        doi = clean_text(doi_match.group(1)) if doi_match else ""

        url_match = re.search(
            r"(https://www\.scopus\.com/pages/publications/\S+)",
            chunk
        )
        url = clean_text(url_match.group(1)) if url_match else ""

        abstract_match = re.search(
            r"ABSTRACT:\s*(.*?)(?=\nAUTHOR KEYWORDS:|\nINDEX KEYWORDS:|\nDOCUMENT TYPE:|\Z)",
            chunk,
            flags=re.DOTALL
        )
        abstract = clean_text(abstract_match.group(1)) if abstract_match else ""

        author_keywords_match = re.search(
            r"AUTHOR KEYWORDS:\s*(.*?)(?=\nINDEX KEYWORDS:|\nDOCUMENT TYPE:|\Z)",
            chunk,
            flags=re.DOTALL
        )
        author_keywords = clean_text(author_keywords_match.group(1)) if author_keywords_match else ""

        index_keywords_match = re.search(
            r"INDEX KEYWORDS:\s*(.*?)(?=\nDOCUMENT TYPE:|\Z)",
            chunk,
            flags=re.DOTALL
        )
        index_keywords = clean_text(index_keywords_match.group(1)) if index_keywords_match else ""

        document_type_match = re.search(
            r"DOCUMENT TYPE:\s*(.*?)(?=\nPUBLICATION STAGE:|\nOPEN ACCESS:|\nSOURCE:|\Z)",
            chunk,
            flags=re.DOTALL
        )
        document_type = clean_text(document_type_match.group(1)) if document_type_match else ""

        keywords = "; ".join(
            x for x in [author_keywords, index_keywords] if x
        )

        papers.append({
            "title": clean_text(title),
            "authors": clean_text(authors),
            "abstract": abstract,
            "keywords": clean_text(keywords),
            "doi": doi,
            "url": url,
            "document_type": document_type,
            "source_info": source_info,
            "citation": clean_text(chunk[:1000])
        })

    unique = []
    seen = set()

    for p in papers:
        key = p["doi"].lower() if p["doi"] else p["title"].lower()
        key = re.sub(r"\s+", " ", key).strip()

        if key and key not in seen:
            unique.append(p)
            seen.add(key)

    print("Raw parsed articles:", len(papers))
    print("Unique articles after deduplication:", len(unique))

    return unique


def extract_json_from_response(text):
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)

    if match:
        return json.loads(match.group(0))

    raise ValueError(f"No valid JSON found in model response: {text[:300]}")


def screen_paper(paper, timeout_seconds=TIMEOUT_SECONDS):

    prompt = f"""
/no_think
You are a strict academic screening assistant.

Base your decision ONLY on the provided title, abstract, and keywords.
Do not use outside knowledge.
Do not invent information.
If evidence is insufficient, choose uncertain.

Return ONLY a JSON object.
Do not include explanations outside JSON.
Do not use markdown.

The JSON must have exactly these fields:
{{
  "include": "include" or "exclude" or "uncertain",
  "main_reason": "short reason based only on the abstract",
  "matched_inclusion_criteria": [],
  "matched_exclusion_criteria": [],
  "paper_type": "original research" or "review/survey" or "unclear",
  "modalities_detected": [],
  "task_detected": "",
  "confidence": "high" or "medium" or "low"
}}

INCLUSION CRITERIA:
{INCLUSION_CRITERIA}

EXCLUSION CRITERIA:
{EXCLUSION_CRITERIA}

Title:
{paper["title"]}

Authors:
{paper["authors"]}

Abstract:
{paper["abstract"]}

Keywords:
{paper["keywords"]}

Document type:
{paper["document_type"]}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 1200
        }
    }

    response = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        timeout=timeout_seconds
    )

    response.raise_for_status()
    result = response.json()

    raw_text = result["message"]["content"]
    print("RAW MODEL RESPONSE:", raw_text[:500])

    return extract_json_from_response(raw_text)


def make_error_decision(error_message):
    return {
        "include": "uncertain",
        "confidence": "low",
        "main_reason": f"Model, timeout, or JSON error: {error_message}",
        "paper_type": "unclear",
        "modalities_detected": [],
        "task_detected": "",
        "matched_inclusion_criteria": [],
        "matched_exclusion_criteria": []
    }


def save_checkpoint(records):
    df_temp = pd.DataFrame(records)
    df_temp.to_csv(CHECKPOINT_CSV, index=False)
    df_temp.to_excel(CHECKPOINT_EXCEL, index=False)


if Path(CHECKPOINT_CSV).exists():
    existing_df = pd.read_csv(CHECKPOINT_CSV)
    all_papers = existing_df.to_dict("records")

    processed_keys = set(
        existing_df["doi"].fillna("").astype(str).str.lower()
        + "||"
        + existing_df["title"].fillna("").astype(str).str.lower()
    )

    print(f"Resuming from checkpoint: {len(all_papers)} papers already processed.")

else:
    all_papers = []
    processed_keys = set()
    print("No checkpoint found. Starting from zero.")


for file in FILES:
    text = Path(file).read_text(encoding="utf-8", errors="ignore")

    print("\n==============================")
    print(f"Reading file: {file}")

    papers = split_scopus_txt(text)

    print(f"{file}: Found {len(papers)} unique articles")

    for i, paper in enumerate(papers, start=1):

        paper_key = paper["doi"].lower() + "||" + paper["title"].lower()

        if paper_key in processed_keys:
            print(f"Skipping already processed paper {i}/{len(papers)}")
            continue

        print(f"\nProcessing {i}/{len(papers)}: {paper['title'][:120]}")

        try:
            decision = screen_paper(paper)

        except Exception as e:
            decision = make_error_decision(str(e))
            print(f"Problem with this paper. Marked uncertain. Error: {e}")

        record = {
            "file": file,
            "title": paper["title"],
            "authors": paper["authors"],
            "source_info": paper["source_info"],
            "doi": paper["doi"],
            "url": paper["url"],
            "document_type": paper["document_type"],
            "abstract": paper["abstract"],
            "keywords": paper["keywords"],
            "decision": decision.get("include", "uncertain"),
            "confidence": decision.get("confidence", "low"),
            "main_reason": decision.get("main_reason", ""),
            "paper_type": decision.get("paper_type", "unclear"),
            "modalities_detected": "; ".join(decision.get("modalities_detected", [])),
            "task_detected": decision.get("task_detected", ""),
            "matched_inclusion_criteria": "; ".join(decision.get("matched_inclusion_criteria", [])),
            "matched_exclusion_criteria": "; ".join(decision.get("matched_exclusion_criteria", []))
        }

        all_papers.append(record)
        processed_keys.add(paper_key)

        save_checkpoint(all_papers)

        print(f"Decision: {record['decision']} | Confidence: {record['confidence']}")
        print(f"Checkpoint saved after {len(all_papers)} total papers.")


df = pd.DataFrame(all_papers)

df.to_excel(OUTPUT_EXCEL, index=False)
df.to_csv(OUTPUT_CSV, index=False)

print("\n==============================")
print(f"Done. Screened {len(df)} papers.")
print(f"Saved final Excel: {OUTPUT_EXCEL}")
print(f"Saved final CSV: {OUTPUT_CSV}")
print(f"Saved checkpoint Excel: {CHECKPOINT_EXCEL}")
print(f"Saved checkpoint CSV: {CHECKPOINT_CSV}")

end = time.perf_counter()
print(f"Execution time: {end - start:.2f} seconds")

No checkpoint found. Starting from zero.

Reading file: scopus_results.txt
Detected article chunks: 372
Detected ABSTRACT fields: 385
Detected DOI fields: 367
Detected Scopus links: 386
Raw parsed articles: 372
Unique articles after deduplication: 372
scopus_results.txt: Found 372 unique articles

Processing 1/372: Tactile Perception in Matched Virtual Reality: Evaluating Virtual Textures on a Real Surface
RAW MODEL RESPONSE: {
  "include": "include",
  "main_reason": "The study involves both visual (virtual textures) and tactile (real surface) modalities, focusing on texture recognition and perception through user judgments",
  "matched_inclusion_criteria": [
    "original research",
    "at least two modalities (vision/tactile)",
    "texture recognition",
    "cross-modal perception in VR"
  ],
  "matched_exclusion_criteria": [],
  "paper_type": "original research",
  "modalities_detected": [
    "vision / visual"
Decision: include | Confidence: high
Checkpoint saved after 1 total p